In [3]:
!pip install pandas pyarrow

Looking in indexes: https://ship-nexus.maestro.gov.sg/repository/pypi-proxy/simple/


In [6]:
import pandas as pd
import pyarrow

# Offline Chunking

simple_preprocess.py → Character-based, offline

- Has built-in simple_chunk_by_chars() function
- Works completely offline, no external dependencies
- Simpler but cuts mid-word/sentence

In [28]:
df = pd.read_parquet("../out/articles.parquet")
df.head(1)

,title,url,news_site,published_date,quarter_year,content,topic,sentiment_score,explanation,year,article_id_key,article_topic_key
0,"19,600 BTO flats to launch this year, includin...",https://www.channelnewsasia.com/singapore/1960...,cna,2025-01-16 17:00:00,2025Q1,Flats with shorter waiting times will become “...,"BTO & SBF Supply, Launches, and Demand",0.8,The article discusses increased BTO supply and...,2025,"19,600 BTO flats to launch this year, includin...","19,600 BTO flats to launch this year, includin..."


## Chunks

In [10]:
chunks = pd.read_parquet("../out/semantic_chunks.parquet")
chunks.head(1)

,article_id_key,article_topic_key,chunk_id,title,topic,year,quarter_year,news_site,published_date,sentiment_score,explanation,content_chunk,retrieval_text
0,"19,600 BTO flats to launch this year, includin...","19,600 BTO flats to launch this year, includin...",0,"19,600 BTO flats to launch this year, includin...","BTO & SBF Supply, Launches, and Demand",2025,2025Q1,cna,2025-01-16 17:00:00,0.8,The article discusses increased BTO supply and...,Flats with shorter waiting times will become “...,"Title: 19,600 BTO flats to launch this year, i..."


In [24]:
print(chunks['content_chunk'].iloc[0])

Flats with shorter waiting times will become “a major feature” of Singapore’s upcoming flat supply, says Minister for National Development Desmond Lee. SINGAPORE: About 19,600 Build-to-Order (BTO) flats will be launched in 2025, as part of the government's efforts to boost public housing supply and meet rising demand. Among them, 3,800 flats – or nearly 20 per cent – will have shorter waiting times of under three years, said Minister for National Development Desmond Lee in an interview with CNA Digital, the Straits Times and Lianhe Zaobao earlier this week. In addition, the Housing Board plans to offermore than 5,500 units in February via its largest-ever Sale of Balance Flats exercise. This brings the total number of flats for sale this year to over 25,000. The flats will includeStandard, Plus and Prime flats – the three categories under arevised classification framework– in locations such as Kallang/Whampoa, Bukit Merah, Queenstown, Mount Pleasant, Woodlands, Yishun and Sembawang. Wi

In [26]:
print(chunks['retrieval_text'].iloc[0])

Title: 19,600 BTO flats to launch this year, including 3,800 units with waiting time of under 3 years
Topic: BTO & SBF Supply, Launches, and Demand
Published date: 2025-01-16 17:00:00
Year: 2025
News site: cna
Sentiment score: 0.8
Topic sentiment explanation: The article discusses increased BTO supply and launches, with positive sentiment towards addressing housing demand and reducing waiting times.

Content:
Flats with shorter waiting times will become “a major feature” of Singapore’s upcoming flat supply, says Minister for National Development Desmond Lee. SINGAPORE: About 19,600 Build-to-Order (BTO) flats will be launched in 2025, as part of the government's efforts to boost public housing supply and meet rising demand. Among them, 3,800 flats – or nearly 20 per cent – will have shorter waiting times of under three years, said Minister for National Development Desmond Lee in an interview with CNA Digital, the Straits Times and Lianhe Zaobao earlier this week. In addition, the Housin

# Token aware chunking

preprocess.py → Sentence-aware + tiktoken

- Uses semantic_chunks.py → calls token_chunking.py → requires tiktoken
- Smart chunking (respects sentences, accurate token counts)
- Need to downloac tiktoken beforehand

In [30]:
chunks = pd.read_parquet("../out2/semantic_chunks.parquet")
chunks.head(1)

,article_id_key,article_topic_key,chunk_id,title,topic,year,quarter_year,news_site,published_date,sentiment_score,explanation,content_chunk,retrieval_text,chunk_token_count,retrieval_token_count
0,"19,600 BTO flats to launch this year, includin...","19,600 BTO flats to launch this year, includin...",0,"19,600 BTO flats to launch this year, includin...","BTO & SBF Supply, Launches, and Demand",2025,2025Q1,cna,2025-01-16 17:00:00,0.8,The article discusses increased BTO supply and...,Flats with shorter waiting times will become “...,"Title: 19,600 BTO flats to launch this year, i...",726,830


In [31]:
print(chunks['content_chunk'].iloc[0])

Flats with shorter waiting times will become “a major feature” of Singapore’s upcoming flat supply, says Minister for National Development Desmond Lee. SINGAPORE: About 19,600 Build-to-Order (BTO) flats will be launched in 2025, as part of the government's efforts to boost public housing supply and meet rising demand. Among them, 3,800 flats – or nearly 20 per cent – will have shorter waiting times of under three years, said Minister for National Development Desmond Lee in an interview with CNA Digital, the Straits Times and Lianhe Zaobao earlier this week. In addition, the Housing Board plans to offermore than 5,500 units in February via its largest-ever Sale of Balance Flats exercise. This brings the total number of flats for sale this year to over 25,000. The flats will includeStandard, Plus and Prime flats – the three categories under arevised classification framework– in locations such as Kallang/Whampoa, Bukit Merah, Queenstown, Mount Pleasant, Woodlands, Yishun and Sembawang. Wi

In [32]:
print(chunks['retrieval_text'].iloc[0])

Title: 19,600 BTO flats to launch this year, including 3,800 units with waiting time of under 3 years
Topic: BTO & SBF Supply, Launches, and Demand
Published date: 2025-01-16
Year: 2025
News site: cna
Sentiment score: 0.8
Topic sentiment explanation: The article discusses increased BTO supply and launches, with positive sentiment towards addressing housing demand and reducing waiting times.

Content:
Flats with shorter waiting times will become “a major feature” of Singapore’s upcoming flat supply, says Minister for National Development Desmond Lee. SINGAPORE: About 19,600 Build-to-Order (BTO) flats will be launched in 2025, as part of the government's efforts to boost public housing supply and meet rising demand. Among them, 3,800 flats – or nearly 20 per cent – will have shorter waiting times of under three years, said Minister for National Development Desmond Lee in an interview with CNA Digital, the Straits Times and Lianhe Zaobao earlier this week. In addition, the Housing Board p